# Retail Sales & Digital Behavior Analytics
**PySpark · Spark SQL · BI layer**

*ES: Analítica de ventas y comportamiento digital con PySpark, Spark SQL y una capa lista para BI.*

---
## 0. Setup

**Requirements / Requisitos:** Python 3.10+, Java 11/17/21, `pip install -r requirements.txt`.
**Data / Datos:** place the five JSON files in `data/` (see `data/README.md`). / coloca los cinco archivos JSON en `data/`.

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, Window

spark = (
    SparkSession.builder
    .appName("retail-sales-analytics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# Locate the data folder whether the notebook runs from /notebooks or from the repo root
DATA_DIR = next((p for p in (Path("../data"), Path("data")) if p.exists()), None)
SOURCES = ["customers.json", "products.json", "orders_jan_mar.json",
           "orders_apr_jun.json", "digital_events.json"]
missing = [f for f in SOURCES if DATA_DIR is None or not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f"Missing files in data/: {missing}")
OUTPUT_DIR = DATA_DIR.parent / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Spark", spark.version, "| data:", DATA_DIR.resolve())

Spark 3.5.6 | data: /home/claude/project/data


# 1. Business context

A commercial company sells across several countries, customer segments and channels (WEB, APP, STORE, CALL_CENTER). During H1 2026 it recorded customers, products, orders and digital events. The BI team needs one trustworthy, integrated view of the business.

**Questions to answer**
1. How did net sales evolve by month, country, category and channel?
2. Which customers and products drive the revenue?
3. How does digital behavior (view → add to cart) relate to completed purchases?

*ES: Una empresa comercial vende en varios países, segmentos y canales. Durante el primer semestre de 2026 registró clientes, productos, órdenes y eventos digitales. BI necesita una visión integrada y confiable. Preguntas: evolución de ventas netas, qué clientes/productos generan el ingreso y cómo se relaciona el comportamiento digital con las compras.*

| Source | Description / Descripción |
|---|---|
| `customers.json` | Customer master / Maestro de clientes |
| `products.json` | Product catalog / Catálogo de productos |
| `orders_jan_mar.json`, `orders_apr_jun.json` | Orders with nested `items` / Órdenes con `items` anidado |
| `digital_events.json` | Clickstream with nested `device`, `attributes`, `items` |

The sources contain deliberate quality issues and nested structures. / Las fuentes contienen problemas de calidad y estructuras anidadas a propósito.

# 2. Ingestion & data quality / Ingesta y calidad de datos

Each source is read independently. For each one: schema, row count, sample and quality profile (duplicates, nulls). The helper below standardizes the profiling.

*ES: Cada fuente se lee de forma independiente: schema, conteo, muestra y perfil de calidad (duplicados, nulos).*

In [2]:
def profile(df, name, key=None):
    """Row count, exact duplicates, repeated keys and null counts per column."""
    n = df.count()
    print(f"[{name}] rows: {n} | exact duplicate rows: {n - df.dropDuplicates().count()}")
    if key:
        repeated = df.groupBy(key).count().filter("count > 1").count()
        print(f"[{name}] '{key}' values appearing more than once: {repeated}")
    nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).first().asDict()
    print(f"[{name}] null counts:", {c: v for c, v in nulls.items() if v} or "none")

## 2.1 Customers

In [3]:
customers_raw = spark.read.json(str(DATA_DIR / "customers.json"))
customers_raw.printSchema()
profile(customers_raw, "customers", key="customer_id")
customers_raw.orderBy("customer_id").show(truncate=False)

root
 |-- active: boolean (nullable = true)
 |-- age: long (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_code: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- signup_date: string (nullable = true)



[customers] rows: 13 | exact duplicate rows: 1


[customers] 'customer_id' values appearing more than once: 1


[customers] null counts: {'age': 1, 'segment': 1}


+------+----+-------+-------------+-----------+--------------+---------+-----------+
|active|age |country|customer_code|customer_id|name          |segment  |signup_date|
+------+----+-------+-------------+-----------+--------------+---------+-----------+
|true  |34  |EC     |C-00101      |101        |Ana Torres    |PREMIUM  |2025-11-12 |
|true  |28  |EC     |C-00102      |102        |Luis Mena     |RETAIL   |2025-12-03 |
|true  |28  |EC     |C-00102      |102        |Luis Mena     |RETAIL   |2025-12-03 |
|true  |41  |CO     |C-00103      |103        |Carla Ruiz    |CORPORATE|2025-09-21 |
|true  |22  |PE     |C-00104      |104        |Mateo Vega    |RETAIL   |2026-01-15 |
|true  |NULL|EC     |C-00105      |105        |Sofía León    |NULL     |2026-02-01 |
|false |37  |CO     |C-00106      |106        |Diego Paz     |PREMIUM  |2025-07-30 |
|true  |45  |US     |C-00107      |107        |Valentina Cruz|CORPORATE|2025-05-18 |
|true  |31  |EC     |C-00108      |108        |Pablo Ortiz   |RET

**Findings / Hallazgos**
- One exact duplicate (`customer_id` 102). / Un duplicado exacto.
- Customer 105 has null `age` and null `segment`. / El cliente 105 tiene `age` y `segment` nulos.
- Names contain accents; `signup_date` is a string. / Nombres con tildes; `signup_date` es string.

In [4]:
ACCENTS_FROM, ACCENTS_TO = "áéíóúñÁÉÍÓÚÑ", "aeiounAEIOUN"

customers_clean = (
    customers_raw
    .dropDuplicates()
    .withColumn("name", F.translate("name", ACCENTS_FROM, ACCENTS_TO))   # native, no Python UDF
    .withColumn("signup_date", F.to_date("signup_date"))
    .withColumn("segment", F.coalesce("segment", F.lit("UNKNOWN")))
)
assert customers_clean.count() == customers_clean.select("customer_id").distinct().count()
customers_clean.orderBy("customer_id").show(truncate=False)

+------+----+-------+-------------+-----------+--------------+---------+-----------+
|active|age |country|customer_code|customer_id|name          |segment  |signup_date|
+------+----+-------+-------------+-----------+--------------+---------+-----------+
|true  |34  |EC     |C-00101      |101        |Ana Torres    |PREMIUM  |2025-11-12 |
|true  |28  |EC     |C-00102      |102        |Luis Mena     |RETAIL   |2025-12-03 |
|true  |41  |CO     |C-00103      |103        |Carla Ruiz    |CORPORATE|2025-09-21 |
|true  |22  |PE     |C-00104      |104        |Mateo Vega    |RETAIL   |2026-01-15 |
|true  |NULL|EC     |C-00105      |105        |Sofia Leon    |UNKNOWN  |2026-02-01 |
|false |37  |CO     |C-00106      |106        |Diego Paz     |PREMIUM  |2025-07-30 |
|true  |45  |US     |C-00107      |107        |Valentina Cruz|CORPORATE|2025-05-18 |
|true  |31  |EC     |C-00108      |108        |Pablo Ortiz   |RETAIL   |2026-03-10 |
|true  |29  |PE     |C-00109      |109        |Maria Solis   |PRE

## 2.2 Products

In [5]:
products_raw = spark.read.json(str(DATA_DIR / "products.json"))
products_raw.printSchema()
profile(products_raw, "products", key="product_id")
products_raw.orderBy("product_id").show(truncate=False)

root
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- list_price: double (nullable = true)
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sku: string (nullable = true)



[products] rows: 10 | exact duplicate rows: 0


[products] 'product_id' values appearing more than once: 0


[products] null counts: none
+---------+-----------+----------+----------+--------------------+-------+
|brand    |category   |list_price|product_id|product_name        |sku    |
+---------+-----------+----------+----------+--------------------+-------+
|Nova     |ELECTRONICS|899.99    |201       |Laptop Nova 14      |SKU-201|
|Atlas    |ELECTRONICS|549.5     |202       |Smartphone Atlas    |SKU-202|
|Brisa    |HOME       |79.9      |203       |Licuadora Brisa     |SKU-203|
|Rayo     |MOTOR      |1899.0    |204       |Motocicleta Rayo 150|SKU-204|
|Polar    |APPLIANCES |699.0     |205       |Refrigeradora Polar |SKU-205|
|Pulse    |ELECTRONICS|59.99     |206       |Audífonos Pulse     |SKU-206|
|Nube     |HOME       |119.0     |207       |Cafetera Nube       |SKU-207|
|Horizonte|ELECTRONICS|329.0     |208       |Tablet Horizonte    |SKU-208|
|Delta    |APPLIANCES |499.0     |209       |Lavadora Delta      |SKU-209|
|Urbano   |MOTOR      |45.0      |210       |Casco Urbano        |SKU-2

**Findings / Hallazgos:** no duplicates or nulls; types are consistent. Only whitespace/case standardization is applied. / Sin duplicados ni nulos; tipos consistentes. Solo se estandarizan espacios y mayúsculas.

In [6]:
products_clean = (
    products_raw
    .withColumn("product_name", F.trim("product_name"))
    .withColumn("brand", F.trim("brand"))
    .withColumn("category", F.upper(F.trim("category")))
)
assert products_clean.count() == products_clean.select("product_id").distinct().count()

## 2.3 Orders (Jan–Mar and Apr–Jun)

In [7]:
orders_jan_mar_raw = spark.read.json(str(DATA_DIR / "orders_jan_mar.json"))
orders_apr_jun_raw = spark.read.json(str(DATA_DIR / "orders_apr_jun.json"))

print("--- Jan-Mar schema ---"); orders_jan_mar_raw.printSchema()
print("--- Apr-Jun schema ---"); orders_apr_jun_raw.printSchema()
profile(orders_jan_mar_raw, "orders_jan_mar", key="order_id")
profile(orders_apr_jun_raw, "orders_apr_jun", key="order_id")

--- Jan-Mar schema ---
root
 |-- channel: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- discount: double (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product_id: long (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |    |    |-- unit_price: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)

--- Apr-Jun schema ---
root
 |-- channel: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product_id: long (nullable = true)
 |    |    |-- quantity: string (nullable = true)
 |    |    |-- unit_price: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_ts: string (null

[orders_jan_mar] rows: 10 | exact duplicate rows: 1


[orders_jan_mar] 'order_id' values appearing more than once: 1


[orders_jan_mar] null counts: {'payment_method': 1}


[orders_apr_jun] rows: 12 | exact duplicate rows: 0


[orders_apr_jun] 'order_id' values appearing more than once: 0


[orders_apr_jun] null counts: none


In [8]:
orders_jan_mar_raw.orderBy("order_id").show(truncate=False)
orders_apr_jun_raw.orderBy("order_id").show(truncate=False)

+-----------+-----------+--------+-----------------------------------+--------+-------------------+--------------+---------+
|channel    |customer_id|discount|items                              |order_id|order_ts           |payment_method|status   |
+-----------+-----------+--------+-----------------------------------+--------+-------------------+--------------+---------+
|WEB        |101        |20.0    |[{201, 1, 899.99}, {206, 2, 59.99}]|O-1001  |2026-01-05T10:15:00|CARD          |COMPLETED|
|APP        |102        |0.0     |[{203, 1, 79.9}]                   |O-1002  |2026-01-08T13:20:00|TRANSFER      |COMPLETED|
|WEB        |103        |0.0     |[{204, 1, 1899.0}]                 |O-1003  |2026-01-11T09:05:00|CARD          |CANCELLED|
|STORE      |101        |50.0    |[{205, 1, 699.0}]                  |O-1004  |2026-02-02T18:45:00|CASH          |COMPLETED|
|WEB        |104        |100.0   |[{204, 1, 1899.0}, {210, 1, 45.0}] |O-1005  |2026-02-09T11:00:00|CARD          |COMPLETED|


+-----------+-----------+--------+------------------+--------+-------------------+--------------+---------+
|channel    |customer_id|discount|items             |order_id|order_ts           |payment_method|status   |
+-----------+-----------+--------+------------------+--------+-------------------+--------------+---------+
|STORE      |109        |0       |[{207, 1, 119.0}] |O-1010  |2026-04-02T10:10:00|CASH          |COMPLETED|
|WEB        |110        |25      |[{205, 1, 699.0}] |O-1011  |2026-04-07T16:50:00|CARD          |COMPLETED|
|APP        |111        |0       |[{201, 1, 899.99}]|O-1012  |2026-04-18T14:05:00|CARD          |COMPLETED|
|WEB        |112        |5       |[{203, 2, 79.9}]  |O-1013  |2026-05-03T09:45:00|TRANSFER      |COMPLETED|
|APP        |102        |15      |[{202, 1, 549.5}] |O-1014  |2026-05-10T19:15:00|CARD          |COMPLETED|
|CALL_CENTER|103        |75      |[{204, 1, 1899.0}]|O-1015  |2026-05-17T11:35:00|CARD          |COMPLETED|
|WEB        |101        |0  

**Findings / Hallazgos**
- Schema drift: in Apr–Jun, `customer_id`, `discount`, `quantity` and `unit_price` arrive as strings (Jan–Mar: numeric). / Deriva de schema: en abr–jun llegan como string.
- `O-1009` is duplicated (Jan–Mar). / `O-1009` está duplicada.
- `O-1006` (RETURNED) has null `payment_method`. / `O-1006` sin método de pago.
- `O-1020`: customer `999` and product `999` do not exist in the masters. / Cliente y producto inexistentes.
- `O-1021`: COMPLETED order with an empty `items` array. / Orden completada sin ítems.
- `order_ts` is a string; `items` is an array of structs. / `order_ts` es string; `items` es array de structs.

In [9]:
ITEMS_TYPE = "array<struct<product_id:bigint,quantity:int,unit_price:double>>"

def normalize_orders(df):
    """Unify types across both order files so they can be safely unioned."""
    return df.select(
        F.trim("order_id").alias("order_id"),
        F.col("customer_id").cast("long").alias("customer_id"),
        F.to_timestamp("order_ts").alias("order_ts"),
        F.upper(F.trim("status")).alias("status"),
        F.upper(F.trim("channel")).alias("channel"),
        F.coalesce(F.upper(F.trim("payment_method")), F.lit("UNKNOWN")).alias("payment_method"),
        F.col("discount").cast("double").alias("discount"),
        F.col("items").cast(ITEMS_TYPE).alias("items"),
    )

orders_jan_mar_clean = normalize_orders(orders_jan_mar_raw)
orders_apr_jun_clean = normalize_orders(orders_apr_jun_raw)
orders_apr_jun_clean.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- discount: double (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product_id: long (nullable = true)
 |    |    |-- quantity: integer (nullable = true)
 |    |    |-- unit_price: double (nullable = true)



## 2.4 Digital events

In [10]:
events_raw = spark.read.json(str(DATA_DIR / "digital_events.json"))
events_raw.printSchema()
profile(events_raw.select("event_id", "event_ts", "customer_id", "event_type",
                          F.col("attributes.campaign").alias("campaign"),
                          F.col("attributes.source").alias("source")), "events", key="event_id")
events_raw.orderBy("event_id").show(truncate=False)

root
 |-- attributes: struct (nullable = true)
 |    |-- campaign: string (nullable = true)
 |    |-- source: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- device: struct (nullable = true)
 |    |-- os: string (nullable = true)
 |    |-- type: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product_id: long (nullable = true)
 |    |    |-- quantity: long (nullable = true)



[events] rows: 15 | exact duplicate rows: 1


[events] 'event_id' values appearing more than once: 1


[events] null counts: {'campaign': 2}
+----------------------+-----------+------------------+--------+-------------------+-----------+----------+
|attributes            |customer_id|device            |event_id|event_ts           |event_type |items     |
+----------------------+-----------+------------------+--------+-------------------+-----------+----------+
|{JAN-LAUNCH, ads}     |101        |{Android, mobile} |E-001   |2026-01-04T09:00:00|view       |[{201, 1}]|
|{JAN-LAUNCH, ads}     |101        |{Android, mobile} |E-002   |2026-01-04T09:05:00|add_to_cart|[{201, 1}]|
|{JAN-LAUNCH, ads}     |101        |{Android, mobile} |E-003   |2026-01-05T10:00:00|purchase   |[{201, 1}]|
|{HOME-WINTER, organic}|102        |{Windows, desktop}|E-004   |2026-01-07T12:00:00|view       |[{203, 1}]|
|{HOME-WINTER, organic}|102        |{Windows, desktop}|E-005   |2026-01-07T12:15:00|add_to_cart|[{203, 1}]|
|{MOTO-FEB, email}     |104        |{iOS, mobile}     |E-006   |2026-02-08T10:00:00|view       |[{

**Findings / Hallazgos**
- `E-010` is duplicated. / `E-010` duplicado.
- `attributes.campaign` has a purely numeric value (`123`) and nulls. / `campaign` con valor numérico (`123`) y nulos.
- `customer_id` is a string; `event_ts` is a string; `device`/`attributes` are structs, `items` an array. / Tipos inconsistentes y estructuras anidadas.

In [11]:
events_typed = events_raw.select(
    F.trim("event_id").alias("event_id"),
    F.coalesce(
        F.to_timestamp("event_ts"),
        F.to_timestamp("event_ts", "dd/MM/yyyy HH:mm:ss"),
    ).alias("event_ts"),
    F.col("customer_id").cast("long").alias("customer_id"),
    F.lower(F.trim("event_type")).alias("event_type"),
    F.lower(F.trim("device.type")).alias("device_type"),
    F.lower(F.trim("device.os")).alias("device_os"),
    # purely numeric or empty campaign codes are invalid -> null
    F.when(F.col("attributes.campaign").rlike(r"^\s*\d*\s*$"), None)
     .otherwise(F.trim("attributes.campaign")).alias("campaign"),
    F.lower(F.trim("attributes.source")).alias("source"),
    "items",
)

# Deterministic de-duplication: keep one row per event_id (latest timestamp)
w_dup = Window.partitionBy("event_id").orderBy(F.col("event_ts").desc_nulls_last())
events_clean = (
    events_typed
    .where(F.col("event_id").isNotNull())
    .withColumn("rn", F.row_number().over(w_dup))
    .where("rn = 1").drop("rn")
)
print("events:", events_raw.count(), "->", events_clean.count())

events: 15 -> 14


## 2.5 Cleaning rules / Reglas de limpieza

*Problem → decision → rationale*

| Problem | Decision | Rationale |
|---|---|---|
| Duplicate customer (102), order (`O-1009`), event (`E-010`) | Drop exact duplicates; events keep one row per `event_id` | Identical records; avoids double-counting revenue and activity |
| Customer 105 with null `age` / `segment` | `segment` → `UNKNOWN`; `age` left null | Neither field is used in any metric; imputing would fabricate data |
| Accents in customer names | Native `translate` normalization | Consistent keys and reporting, no Python UDF overhead |
| Date/timestamp columns as strings; Apr–Jun numeric fields as strings | Explicit casts to `date`/`timestamp`/numeric | Safe union of both order files and time-based analysis |
| Null `payment_method` (`O-1006`) | → `UNKNOWN` | Order is RETURNED, so it does not affect revenue; kept for traceability |
| Order with unknown customer and product (`O-1020`) | Quarantined in `orders_rejected` | Revenue cannot be attributed to any customer or catalog item |
| COMPLETED order with empty `items` (`O-1021`) | Quarantined in `orders_rejected` | No revenue to recognize; likely a system bug worth reporting upstream |
| Unknown `product_id` inside a valid order | Line dropped from the detail | Cannot be classified by category |
| CANCELLED / RETURNED orders | Kept, net sales = 0 | Preserves status analysis without inflating revenue |
| Numeric campaign code (`123`) | → null | Not a valid campaign identifier |
| Discount is defined per order, not per line | Allocated to lines proportionally to line subtotal | Enables net sales by product/category that reconcile to the order total |

*ES — Resumen de decisiones:* se eliminan duplicados exactos; `segment` nulo pasa a `UNKNOWN` y `age` se deja nulo porque no se usa en ninguna métrica; se normalizan tildes con funciones nativas; se convierten tipos de forma explícita; las órdenes sin cliente/producto válido o completadas sin ítems se envían a `orders_rejected` (cuarentena) para no inflar ventas; las órdenes canceladas/devueltas se conservan con venta neta 0; los productos inexistentes se eliminan del detalle; la campaña numérica se anula; y el descuento por orden se prorratea entre las líneas según su subtotal.

# 3. Spark transformations / Transformaciones Spark

## 3.1 Consolidate orders / Consolidar órdenes

In [12]:
orders_clean = (
    orders_jan_mar_clean
    .unionByName(orders_apr_jun_clean)
    .dropDuplicates()
)
assert orders_clean.count() == orders_clean.select("order_id").distinct().count(), "order_id must be unique"
print("orders:", orders_jan_mar_raw.count() + orders_apr_jun_raw.count(), "->", orders_clean.count())

orders: 22 -> 21


## 3.2 Order lines / Detalle de órdenes

In [13]:
order_lines = (
    orders_clean
    .select("order_id", "customer_id", "order_ts", "status", "channel",
            F.explode("items").alias("item"))          # explode drops orders with empty arrays
    .select("order_id", "customer_id", "order_ts", "status", "channel",
            F.col("item.product_id").alias("product_id"),
            F.col("item.quantity").alias("quantity"),
            F.col("item.unit_price").alias("unit_price"))
    .withColumn("subtotal", F.col("quantity") * F.col("unit_price"))
)
order_lines.orderBy("order_id", "product_id").show(40, truncate=False)

+--------+-----------+-------------------+---------+-----------+----------+--------+----------+--------+
|order_id|customer_id|order_ts           |status   |channel    |product_id|quantity|unit_price|subtotal|
+--------+-----------+-------------------+---------+-----------+----------+--------+----------+--------+
|O-1001  |101        |2026-01-05 10:15:00|COMPLETED|WEB        |201       |1       |899.99    |899.99  |
|O-1001  |101        |2026-01-05 10:15:00|COMPLETED|WEB        |206       |2       |59.99     |119.98  |
|O-1002  |102        |2026-01-08 13:20:00|COMPLETED|APP        |203       |1       |79.9      |79.9    |
|O-1003  |103        |2026-01-11 09:05:00|CANCELLED|WEB        |204       |1       |1899.0    |1899.0  |
|O-1004  |101        |2026-02-02 18:45:00|COMPLETED|STORE      |205       |1       |699.0     |699.0   |
|O-1005  |104        |2026-02-09 11:00:00|COMPLETED|WEB        |204       |1       |1899.0    |1899.0  |
|O-1005  |104        |2026-02-09 11:00:00|COMPLETED|WEB

## 3.3 Net sales / Ventas netas

`net sales = line subtotals − order discount`, recognized only for COMPLETED orders. Orders that fail validation go to quarantine.

*ES: venta neta = subtotal de líneas − descuento; solo se reconoce en órdenes COMPLETED. Las órdenes inválidas van a cuarentena.*

In [14]:
# 1) Keep only lines whose product exists in the catalog
valid_lines = order_lines.join(products_clean.select("product_id"), "product_id", "left_semi")
order_subtotals = valid_lines.groupBy("order_id").agg(F.sum("subtotal").alias("gross_subtotal"))

# 2) Validate orders and assign a rejection reason
orders_checked = (
    orders_clean
    .join(order_subtotals, "order_id", "left")
    .join(customers_clean.select("customer_id", F.lit(True).alias("customer_found")), "customer_id", "left")
    .withColumn("reject_reason", F.concat_ws("; ",
        F.when(F.size("items") == 0, "EMPTY_ITEMS"),
        F.when((F.size("items") > 0) & F.col("gross_subtotal").isNull(), "UNKNOWN_PRODUCT"),
        F.when(F.col("customer_found").isNull(), "UNKNOWN_CUSTOMER")))
    .withColumn("reject_reason", F.when(F.col("reject_reason") == "", None).otherwise(F.col("reject_reason")))
)

orders_rejected = (
    orders_checked.where("reject_reason IS NOT NULL")
    .select("order_id", "customer_id", "status", "channel", "order_ts", "reject_reason")
)
print("Quarantined orders / Órdenes en cuarentena:")
orders_rejected.show(truncate=False)

# 3) Order-level fact table
fact_orders = (
    orders_checked.where("reject_reason IS NULL")
    .withColumn("order_date", F.to_date("order_ts"))
    .withColumn("net_total",
        F.when(F.col("status") == "COMPLETED", F.col("gross_subtotal") - F.col("discount"))
         .otherwise(F.lit(0.0)))
    .select("order_id", "customer_id", "order_ts", "order_date", "status", "channel",
            "payment_method", "discount", "gross_subtotal", "net_total")
)
fact_orders.orderBy("order_id").show(30, truncate=False)

Quarantined orders / Órdenes en cuarentena:


+--------+-----------+---------+-------+-------------------+---------------------------------+
|order_id|customer_id|status   |channel|order_ts           |reject_reason                    |
+--------+-----------+---------+-------+-------------------+---------------------------------+
|O-1020  |999        |COMPLETED|WEB    |2026-06-22 10:00:00|UNKNOWN_PRODUCT; UNKNOWN_CUSTOMER|
|O-1021  |110        |COMPLETED|WEB    |2026-06-25 10:00:00|EMPTY_ITEMS                      |
+--------+-----------+---------+-------+-------------------+---------------------------------+



+--------+-----------+-------------------+----------+---------+-----------+--------------+--------+--------------+---------+
|order_id|customer_id|order_ts           |order_date|status   |channel    |payment_method|discount|gross_subtotal|net_total|
+--------+-----------+-------------------+----------+---------+-----------+--------------+--------+--------------+---------+
|O-1001  |101        |2026-01-05 10:15:00|2026-01-05|COMPLETED|WEB        |CARD          |20.0    |1019.97       |999.97   |
|O-1002  |102        |2026-01-08 13:20:00|2026-01-08|COMPLETED|APP        |TRANSFER      |0.0     |79.9          |79.9     |
|O-1003  |103        |2026-01-11 09:05:00|2026-01-11|CANCELLED|WEB        |CARD          |0.0     |1899.0        |0.0      |
|O-1004  |101        |2026-02-02 18:45:00|2026-02-02|COMPLETED|STORE      |CASH          |50.0    |699.0         |649.0    |
|O-1005  |104        |2026-02-09 11:00:00|2026-02-09|COMPLETED|WEB        |CARD          |100.0   |1944.0        |1844.0   |


In [15]:
# 4) Line-level fact table: discount allocated proportionally to each line's subtotal
fact_order_lines = (
    valid_lines
    .join(fact_orders.select("order_id", "gross_subtotal", "discount"), "order_id")
    .join(products_clean.select("product_id", "product_name", "category"), "product_id")
    .withColumn("allocated_discount", F.col("discount") * F.col("subtotal") / F.col("gross_subtotal"))
    .withColumn("net_line",
        F.when(F.col("status") == "COMPLETED", F.col("subtotal") - F.col("allocated_discount"))
         .otherwise(F.lit(0.0)))
    .select("order_id", "customer_id", "order_ts", "status", "channel", "product_id",
            "product_name", "category", "quantity", "unit_price", "subtotal",
            "allocated_discount", "net_line")
)

# Reconciliation test: line-level net sales must add up to the order-level net total
recon = (fact_order_lines.groupBy("order_id").agg(F.sum("net_line").alias("lines_net"))
         .join(fact_orders.select("order_id", "net_total"), "order_id")
         .where(F.abs(F.col("lines_net") - F.col("net_total")) > 0.005))
assert recon.count() == 0, "line-level and order-level net sales do not reconcile"
print("Reconciliation OK: lines add up to orders")

Reconciliation OK: lines add up to orders


## 3.4 Customer metrics / Métricas por cliente

In [16]:
categories = (fact_order_lines.where("status = 'COMPLETED'")
              .groupBy("customer_id").agg(F.countDistinct("category").alias("categories_bought")))

customer_metrics = (
    fact_orders.where("status = 'COMPLETED'")
    .groupBy("customer_id")
    .agg(F.countDistinct("order_id").alias("orders_completed"),
         F.round(F.sum("net_total"), 2).alias("net_sales"),
         F.round(F.avg("net_total"), 2).alias("avg_ticket"),
         F.min("order_date").alias("first_purchase"),
         F.max("order_date").alias("last_purchase"))
    .join(categories, "customer_id", "left")
    .join(customers_clean.select("customer_id", "name", "country"), "customer_id", "left")
    .select("customer_id", "name", "country", "orders_completed", "net_sales", "avg_ticket",
            "first_purchase", "last_purchase", "categories_bought")
    .orderBy(F.desc("net_sales"))
)
customer_metrics.show(truncate=False)

+-----------+--------------+-------+----------------+---------+----------+--------------+-------------+-----------------+
|customer_id|name          |country|orders_completed|net_sales|avg_ticket|first_purchase|last_purchase|categories_bought|
+-----------+--------------+-------+----------------+---------+----------+--------------+-------------+-----------------+
|101        |Ana Torres    |EC     |3               |2198.47  |732.82    |2026-01-05    |2026-06-01   |2                |
|104        |Mateo Vega    |PE     |2               |2153.0   |1076.5    |2026-02-09    |2026-06-09   |2                |
|103        |Carla Ruiz    |CO     |1               |1824.0   |1824.0    |2026-05-17    |2026-05-17   |1                |
|108        |Pablo Ortiz   |EC     |2               |1098.49  |549.25    |2026-03-19    |2026-06-15   |2                |
|111        |Elena Mora    |EC     |1               |899.99   |899.99    |2026-04-18    |2026-04-18   |1                |
|110        |Jorge Rivas

## 3.5 Window functions / Funciones de ventana

In [17]:
w_cust = Window.partitionBy("customer_id").orderBy("order_ts", "order_id")

purchase_sequence = (
    fact_orders.where("status = 'COMPLETED'")
    .select("customer_id", "order_id", "order_ts", "order_date")
    .withColumn("previous_order", F.lag("order_id").over(w_cust))
    .withColumn("previous_order_date", F.lag("order_date").over(w_cust))
    .withColumn("days_since_previous", F.datediff("order_date", "previous_order_date"))
    .withColumnRenamed("order_id", "current_order")
    .drop("order_ts")
    .orderBy("customer_id", "order_date")
)
purchase_sequence.show(30, truncate=False)

+-----------+-------------+----------+--------------+-------------------+-------------------+
|customer_id|current_order|order_date|previous_order|previous_order_date|days_since_previous|
+-----------+-------------+----------+--------------+-------------------+-------------------+
|101        |O-1001       |2026-01-05|NULL          |NULL               |NULL               |
|101        |O-1004       |2026-02-02|O-1001        |2026-01-05         |28                 |
|101        |O-1016       |2026-06-01|O-1004        |2026-02-02         |119                |
|102        |O-1002       |2026-01-08|NULL          |NULL               |NULL               |
|102        |O-1014       |2026-05-10|O-1002        |2026-01-08         |122                |
|103        |O-1015       |2026-05-17|NULL          |NULL               |NULL               |
|104        |O-1005       |2026-02-09|NULL          |NULL               |NULL               |
|104        |O-1017       |2026-06-09|O-1005        |2026-02

## 3.6 Top 3 products per country / Top 3 productos por país

In [18]:
w_rank = Window.partitionBy("country").orderBy(F.desc("net_sales"), F.asc("product_id"))   # deterministic tie-break

top3_products_by_country = (
    fact_order_lines.where("status = 'COMPLETED'")
    .join(customers_clean.select("customer_id", "country"), "customer_id")
    .groupBy("country", "product_id", "product_name")
    .agg(F.round(F.sum("net_line"), 2).alias("net_sales"))
    .withColumn("rank", F.row_number().over(w_rank))
    .where("rank <= 3")
    .orderBy("country", "rank")
)
top3_products_by_country.show(truncate=False)

+-------+----------+--------------------+---------+----+
|country|product_id|product_name        |net_sales|rank|
+-------+----------+--------------------+---------+----+
|CO     |204       |Motocicleta Rayo 150|1824.0   |1   |
|CO     |205       |Refrigeradora Polar |674.0    |2   |
|CO     |209       |Lavadora Delta      |499.0    |3   |
|EC     |202       |Smartphone Atlas    |2173.98  |1   |
|EC     |201       |Laptop Nova 14      |1782.33  |2   |
|EC     |205       |Refrigeradora Polar |649.0    |3   |
|MX     |203       |Licuadora Brisa     |154.8    |1   |
|PE     |204       |Motocicleta Rayo 150|1801.31  |1   |
|PE     |208       |Tablet Horizonte    |309.0    |2   |
|PE     |207       |Cafetera Nube       |119.0    |3   |
|US     |208       |Tablet Horizonte    |628.0    |1   |
+-------+----------+--------------------+---------+----+



# 4. Nested data & digital events / Datos anidados y eventos

Structs (`device`, `attributes`) were flattened and types normalized in section 2. Here the `items` array is exploded with `explode_outer`, so events without products are preserved.

*ES: los structs se aplanaron en la sección 2; aquí se explota `items` con `explode_outer` para conservar eventos sin productos.*

In [19]:
events_normalized = (
    events_clean
    .withColumn("item", F.explode_outer("items"))
    .select("event_id", "event_ts", "customer_id", "event_type", "device_type", "device_os",
            F.col("item.product_id").alias("product_id"),
            F.col("item.quantity").alias("quantity"),
            "campaign", "source")
)
events_normalized.printSchema()
events_normalized.orderBy("event_ts").show(truncate=False)

root
 |-- event_id: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- device_os: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- campaign: string (nullable = true)
 |-- source: string (nullable = true)



+--------+-------------------+-----------+-----------+-----------+---------+----------+--------+-----------+-------+
|event_id|event_ts           |customer_id|event_type |device_type|device_os|product_id|quantity|campaign   |source |
+--------+-------------------+-----------+-----------+-----------+---------+----------+--------+-----------+-------+
|E-001   |2026-01-04 09:00:00|101        |view       |mobile     |android  |201       |1       |JAN-LAUNCH |ads    |
|E-002   |2026-01-04 09:05:00|101        |add_to_cart|mobile     |android  |201       |1       |JAN-LAUNCH |ads    |
|E-003   |2026-01-05 10:00:00|101        |purchase   |mobile     |android  |201       |1       |JAN-LAUNCH |ads    |
|E-004   |2026-01-07 12:00:00|102        |view       |desktop    |windows  |203       |1       |HOME-WINTER|organic|
|E-005   |2026-01-07 12:15:00|102        |add_to_cart|desktop    |windows  |203       |1       |HOME-WINTER|organic|
|E-006   |2026-02-08 10:00:00|104        |view       |mobile    

# 5. Spark SQL

All queries run on the curated tables (`fact_orders`, `fact_order_lines`, `customers`, `products`, `events`). Net sales come from the line-level allocation, so they reconcile with order totals.

*ES: todas las consultas usan las tablas curadas; las ventas netas por línea cuadran con el total de la orden.*

In [20]:
fact_orders.createOrReplaceTempView("fact_orders")
fact_order_lines.createOrReplaceTempView("fact_order_lines")
customers_clean.createOrReplaceTempView("customers")
products_clean.createOrReplaceTempView("products")
events_normalized.createOrReplaceTempView("events")

### SQL 1 — Net sales by country, month and category

In [21]:
spark.sql("""
    SELECT c.country,
           date_format(l.order_ts, 'yyyy-MM') AS month,
           l.category,
           ROUND(SUM(l.net_line), 2)          AS net_sales
    FROM fact_order_lines l
    JOIN customers c ON c.customer_id = l.customer_id
    WHERE l.status = 'COMPLETED'
    GROUP BY c.country, date_format(l.order_ts, 'yyyy-MM'), l.category
    ORDER BY c.country, month, l.category
""").show(50, truncate=False)

+-------+-------+-----------+---------+
|country|month  |category   |net_sales|
+-------+-------+-----------+---------+
|CO     |2026-03|APPLIANCES |499.0    |
|CO     |2026-04|APPLIANCES |674.0    |
|CO     |2026-05|MOTOR      |1824.0   |
|EC     |2026-01|ELECTRONICS|999.97   |
|EC     |2026-01|HOME       |79.9     |
|EC     |2026-02|APPLIANCES |649.0    |
|EC     |2026-03|ELECTRONICS|599.49   |
|EC     |2026-04|ELECTRONICS|899.99   |
|EC     |2026-05|ELECTRONICS|534.5    |
|EC     |2026-06|APPLIANCES |499.0    |
|EC     |2026-06|ELECTRONICS|1099.0   |
|MX     |2026-05|HOME       |154.8    |
|PE     |2026-02|MOTOR      |1844.0   |
|PE     |2026-04|HOME       |119.0    |
|PE     |2026-06|ELECTRONICS|309.0    |
|US     |2026-03|ELECTRONICS|628.0    |
+-------+-------+-----------+---------+



### SQL 2 — Customers spending above their country average

In [22]:
spark.sql("""
    WITH spend AS (
        SELECT c.customer_id, c.name, c.country, SUM(o.net_total) AS net_spend
        FROM fact_orders o
        JOIN customers c ON c.customer_id = o.customer_id
        WHERE o.status = 'COMPLETED'
        GROUP BY c.customer_id, c.name, c.country
    ),
    with_avg AS (
        SELECT *, AVG(net_spend) OVER (PARTITION BY country) AS country_avg
        FROM spend
    )
    SELECT name, country, ROUND(net_spend, 2) AS net_spend, ROUND(country_avg, 2) AS country_avg
    FROM with_avg
    WHERE net_spend > country_avg
    ORDER BY country, net_spend DESC
""").show(truncate=False)

+-----------+-------+---------+-----------+
|name       |country|net_spend|country_avg|
+-----------+-------+---------+-----------+
|Carla Ruiz |CO     |1824.0   |999.0      |
|Ana Torres |EC     |2198.47  |1072.17    |
|Pablo Ortiz|EC     |1098.49  |1072.17    |
|Mateo Vega |PE     |2153.0   |1136.0     |
+-----------+-------+---------+-----------+



### SQL 3 — Product ranking by revenue within each country

In [23]:
spark.sql("""
    SELECT country, product_name, net_sales,
           ROW_NUMBER() OVER (PARTITION BY country ORDER BY net_sales DESC, product_id) AS ranking
    FROM (
        SELECT c.country, l.product_id, l.product_name, ROUND(SUM(l.net_line), 2) AS net_sales
        FROM fact_order_lines l
        JOIN customers c ON c.customer_id = l.customer_id
        WHERE l.status = 'COMPLETED'
        GROUP BY c.country, l.product_id, l.product_name
    )
    ORDER BY country, ranking
""").show(50, truncate=False)

+-------+--------------------+---------+-------+
|country|product_name        |net_sales|ranking|
+-------+--------------------+---------+-------+
|CO     |Motocicleta Rayo 150|1824.0   |1      |
|CO     |Refrigeradora Polar |674.0    |2      |
|CO     |Lavadora Delta      |499.0    |3      |
|EC     |Smartphone Atlas    |2173.98  |1      |
|EC     |Laptop Nova 14      |1782.33  |2      |
|EC     |Refrigeradora Polar |649.0    |3      |
|EC     |Lavadora Delta      |499.0    |4      |
|EC     |Audífonos Pulse     |176.63   |5      |
|EC     |Licuadora Brisa     |79.9     |6      |
|MX     |Licuadora Brisa     |154.8    |1      |
|PE     |Motocicleta Rayo 150|1801.31  |1      |
|PE     |Tablet Horizonte    |309.0    |2      |
|PE     |Cafetera Nube       |119.0    |3      |
|PE     |Casco Urbano        |42.69    |4      |
|US     |Tablet Horizonte    |628.0    |1      |
+-------+--------------------+---------+-------+



### SQL 4 — Net sales by channel (pivot)

In [24]:
spark.sql("""
    SELECT country,
           COALESCE(WEB, 0)         AS WEB,
           COALESCE(APP, 0)         AS APP,
           COALESCE(STORE, 0)       AS STORE,
           COALESCE(CALL_CENTER, 0) AS CALL_CENTER
    FROM (
        SELECT c.country, o.channel, o.net_total
        FROM fact_orders o
        JOIN customers c ON c.customer_id = o.customer_id
        WHERE o.status = 'COMPLETED'
    )
    PIVOT (
        ROUND(SUM(net_total), 2)
        FOR channel IN ('WEB' AS WEB, 'APP' AS APP, 'STORE' AS STORE, 'CALL_CENTER' AS CALL_CENTER)
    )
    ORDER BY country
""").show(truncate=False)

+-------+-------+-------+-----+-----------+
|country|WEB    |APP    |STORE|CALL_CENTER|
+-------+-------+-------+-----+-----------+
|CO     |674.0  |0.0    |0.0  |2323.0     |
|EC     |2048.47|2663.38|649.0|0.0        |
|MX     |154.8  |0.0    |0.0  |0.0        |
|PE     |1844.0 |309.0  |119.0|0.0        |
|US     |628.0  |0.0    |0.0  |0.0        |
+-------+-------+-------+-----+-----------+



### SQL 5 — Abandoned carts

Customers with a `view` followed by an `add_to_cart` of the **same product**, and **no completed order for that product on or after the cart event**.

*ES: clientes con `view` seguido de `add_to_cart` del mismo producto y sin compra completada de ese producto posterior al carrito.*

In [25]:
spark.sql("""
    WITH cart_after_view AS (
        SELECT DISTINCT a.customer_id, a.product_id, a.event_ts AS cart_ts
        FROM events v
        JOIN events a
          ON  a.customer_id = v.customer_id
          AND a.product_id  = v.product_id
          AND v.event_type  = 'view'
          AND a.event_type  = 'add_to_cart'
          AND a.event_ts    > v.event_ts
    )
    SELECT c.customer_id, c.name, p.product_name, x.cart_ts
    FROM cart_after_view x
    JOIN customers c ON c.customer_id = x.customer_id
    JOIN products  p ON p.product_id  = x.product_id
    WHERE NOT EXISTS (
        SELECT 1
        FROM fact_order_lines l
        WHERE l.customer_id = x.customer_id
          AND l.product_id  = x.product_id
          AND l.status      = 'COMPLETED'
          AND l.order_ts   >= x.cart_ts
    )
    ORDER BY x.cart_ts
""").show(truncate=False)

+-----------+----------+----------------+-------------------+
|customer_id|name      |product_name    |cart_ts            |
+-----------+----------+----------------+-------------------+
|105        |Sofia Leon|Smartphone Atlas|2026-06-21 11:00:00|
+-----------+----------+----------------+-------------------+



# 6. Analytics layer for BI / Capa analítica para BI

The curated tables are exported as CSV to `output/` and loaded into Power BI to build the dashboard (screenshots below). The KPIs are also computed here in Spark so the dashboard can be validated against them.

*ES: las tablas curadas se exportan a `output/` y se cargan en Power BI para construir el dashboard. Los KPIs se calculan aquí para validar el tablero.*

In [26]:
EXPORTS = {
    "fact_orders": fact_orders,
    "fact_order_lines": fact_order_lines,
    "fact_events": events_normalized,
    "dim_customers": customers_clean,
    "dim_products": products_clean,
    "customer_metrics": customer_metrics,
}
for name, df in EXPORTS.items():
    df.toPandas().to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
print("Exported:", sorted(p.name for p in OUTPUT_DIR.glob("*.csv")))

Exported: ['customer_metrics.csv', 'dim_customers.csv', 'dim_products.csv', 'fact_events.csv', 'fact_order_lines.csv', 'fact_orders.csv']


## 6.1 KPIs

In [27]:
spark.sql("""
    SELECT COUNT(*)                      AS orders_completed,
           ROUND(SUM(net_total), 2)      AS net_sales,
           ROUND(AVG(net_total), 2)      AS avg_ticket,
           COUNT(DISTINCT customer_id)   AS buyers
    FROM fact_orders
    WHERE status = 'COMPLETED'
""").show()

+----------------+---------+----------+------+
|orders_completed|net_sales|avg_ticket|buyers|
+----------------+---------+----------+------+
|              17| 11412.65|    671.33|    12|
+----------------+---------+----------+------+



## 6.2 Monthly net sales, by channel and digital funnel

In [28]:
spark.sql("""
    SELECT date_format(order_date, 'yyyy-MM') AS month,
           COUNT(*)                           AS orders,
           ROUND(SUM(net_total), 2)           AS net_sales
    FROM fact_orders WHERE status = 'COMPLETED'
    GROUP BY 1 ORDER BY 1
""").show()

spark.sql("""
    SELECT channel,
           ROUND(SUM(net_total), 2) AS net_sales,
           ROUND(100 * SUM(net_total) / SUM(SUM(net_total)) OVER (), 1) AS pct_of_total
    FROM fact_orders WHERE status = 'COMPLETED'
    GROUP BY channel ORDER BY net_sales DESC
""").show()

+-------+------+---------+
|  month|orders|net_sales|
+-------+------+---------+
|2026-01|     2|  1079.87|
|2026-02|     2|   2493.0|
|2026-03|     3|  1726.49|
|2026-04|     3|  1692.99|
|2026-05|     3|   2513.3|
|2026-06|     4|   1907.0|
+-------+------+---------+



+-----------+---------+------------+
|    channel|net_sales|pct_of_total|
+-----------+---------+------------+
|        WEB|  5349.27|        46.9|
|        APP|  2972.38|        26.0|
|CALL_CENTER|   2323.0|        20.4|
|      STORE|    768.0|         6.7|
+-----------+---------+------------+



In [29]:
# Digital activity vs purchases, per customer
spark.sql("""
    WITH ev AS (
        SELECT customer_id,
               SUM(CASE WHEN event_type = 'view'        THEN 1 ELSE 0 END) AS views,
               SUM(CASE WHEN event_type = 'add_to_cart' THEN 1 ELSE 0 END) AS add_to_cart,
               SUM(CASE WHEN event_type = 'purchase'    THEN 1 ELSE 0 END) AS purchase_events
        FROM (SELECT DISTINCT event_id, customer_id, event_type FROM events)
        GROUP BY customer_id
    ),
    od AS (
        SELECT customer_id, COUNT(*) AS completed_orders, ROUND(SUM(net_total), 2) AS net_sales
        FROM fact_orders WHERE status = 'COMPLETED' GROUP BY customer_id
    )
    SELECT c.name, ev.views, ev.add_to_cart, ev.purchase_events,
           COALESCE(od.completed_orders, 0) AS completed_orders,
           COALESCE(od.net_sales, 0)        AS net_sales
    FROM ev
    JOIN customers c ON c.customer_id = ev.customer_id
    LEFT JOIN od     ON od.customer_id = ev.customer_id
    ORDER BY ev.views DESC, c.name
""").show(truncate=False)

+-----------+-----+-----------+---------------+----------------+---------+
|name       |views|add_to_cart|purchase_events|completed_orders|net_sales|
+-----------+-----+-----------+---------------+----------------+---------+
|Ana Torres |1    |1          |1              |3               |2198.47  |
|Luis Mena  |1    |1          |0              |2               |614.4    |
|Maria Solis|1    |1          |0              |1               |119.0    |
|Mateo Vega |1    |1          |1              |2               |2153.0   |
|Pablo Ortiz|1    |1          |0              |2               |1098.49  |
|Sofia Leon |1    |1          |0              |1               |549.5    |
+-----------+-----+-----------+---------------+----------------+---------+



## 6.3 Dashboard (Power BI)

> Screenshots to be added after building the report in Power BI. / Capturas por agregar tras construir el reporte en Power BI.

**Page 1 — Executive overview**
- KPI cards: completed orders, net sales, average ticket, buyers.
- Line chart: monthly net sales.
- Bar chart: net sales by channel.

![Executive overview](../images/dashboard_overview.png)

**Page 2 — Digital behavior vs purchases**
- Funnel / table: views → add to cart → completed purchase, per customer.
- Abandoned-cart list (SQL 5).

![Digital behavior](../images/dashboard_digital.png)

Data source: CSV files in `output/`. Relationships: `fact_orders[customer_id]` → `dim_customers`, `fact_order_lines[product_id]` → `dim_products`, `fact_order_lines[order_id]` → `fact_orders`.

## 6.4 Insights & storytelling

**Three conclusions / Tres conclusiones**
1. **Revenue is concentrated by channel and country.** WEB generates 46.9% of net sales (5,349.27 of 11,412.65) and Ecuador 47.0% (5,360.85). / *WEB concentra el 46.9% de la venta neta y Ecuador el 47.0%.*
2. **Two high-ticket orders drive a third of revenue.** The two motorcycle orders (`O-1005`, `O-1015`) bring 3,668.00 (32.1%) from 11.8% of completed orders; Electronics is the largest category (44.4%). / *Dos órdenes de motocicletas aportan 32.1% de la venta neta con 11.8% de las órdenes.*
3. **Digital intent converts, but tracking is incomplete.** 5 of the 6 customers with a view → add-to-cart sequence later bought the same product (83%). However, only 2 of those 5 purchases have a `purchase` event, so event tracking under-reports conversions. / *5 de 6 clientes con vista → carrito compraron el producto (83%), pero solo 2 de esas 5 compras tienen evento `purchase`: el tracking subestima conversiones.*

**Finding that requires investigation / Hallazgo a investigar**

Net sales peak in the middle month of each quarter (Feb 2,493.00; May 2,513.30, the semester high) and every quarter closes above its opening month. But one motorcycle order explains 74% of February and 73% of May. Without those two orders, February (649.00) and May (689.30) would be the weakest months of their quarters.

*Los picos de mitad de trimestre (feb 2,493.00; may 2,513.30, el máximo del semestre) dependen de una orden de motocicleta cada uno (74% y 73% del mes). Sin ellas, feb y may serían los meses más bajos de su trimestre.*

Is it recurring seasonality/campaign effect or just two large one-off sales? February has a matching campaign (`MOTO-FEB`, email); May's order (call center) has no campaign recorded. Next step: longer history and order-level campaign attribution.

**KPIs to track for the business question / KPIs recomendados**

| KPI | Definition | Why it matters |
|---|---|---|
| Net sales and average ticket | `SUM(net_total)` / completed orders (11,412.65; 671.33) | Core revenue health; average ticket is sensitive to a few large orders, so report it with the median |
| Cart-to-purchase conversion | Customers who bought the product after `add_to_cart` / customers with `add_to_cart` | Connects digital behavior to sales; requires fixing `purchase` event tracking |
| Repeat purchase rate | Customers with 2+ completed orders / buyers (4 of 12 = 33%) | Shows loyalty; average gap between purchases is ~95 days |
| Cancellation + return rate | CANCELLED + RETURNED / valid orders (2 of 19 = 10.5%) | Revenue leakage that net sales hides |
| Discount rate | Discounts / gross subtotal on completed orders (2.98%) | Measures margin pressure |